In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Load the cleaned data you saved yesterday
df = pd.read_csv('../outputs/hle_at_birth_cleaned.csv')

print(f"✓ Loaded {len(df)} rows")
print("\nData preview:")
print(df.head())

✓ Loaded 7326 rows

Data preview:
         Period  Country    Area type  Area code             Area name  \
0  2011 to 2013  England  Local Areas  E06000001            Hartlepool   
1  2011 to 2013  England  Local Areas  E06000001            Hartlepool   
2  2011 to 2013  England  Local Areas  E06000002         Middlesbrough   
3  2011 to 2013  England  Local Areas  E06000002         Middlesbrough   
4  2011 to 2013  England  Local Areas  E06000003  Redcar and Cleveland   

      Sex  Sex code Age group  Age code   HLE   LCI   UCI  Proportion (%)  \
0    Male         1        <1         1  56.1  54.0  58.2              72   
1  Female         2        <1         1  57.3  54.9  59.7              70   
2    Male         1        <1         1  57.7  55.6  59.8              75   
3  Female         2        <1         1  58.8  56.7  60.9              73   
4    Male         1        <1         1  58.9  56.7  61.2              75   

   LE_calculated         LE    HLE_Gap  
0      77.916667 

In [2]:
# Calculate the average of Male and Female to get "Persons"
# Group by Area and Period, then average the HLE, LE, and Gap

df_persons = df.groupby(['Area name', 'Area code', 'Period']).agg({
    'HLE': 'mean',
    'LE': 'mean',
    'HLE_Gap': 'mean'
}).reset_index()

print(f"✓ Created 'Persons' data: {len(df_persons)} rows")
print(f"✓ This is {len(df_persons['Area name'].unique())} local authorities × {len(df_persons['Period'].unique())} time periods")

print("\nSample:")
print(df_persons.head(10))

✓ Created 'Persons' data: 3663 rows
✓ This is 308 local authorities × 12 time periods

Sample:
       Area name  Area code        Period    HLE         LE    HLE_Gap
0  Aberdeen City  S12000033  2011 to 2013  64.10  79.164634  15.064634
1  Aberdeen City  S12000033  2012 to 2014  64.55  78.746839  14.196839
2  Aberdeen City  S12000033  2013 to 2015  64.75  78.989290  14.239290
3  Aberdeen City  S12000033  2014 to 2016  63.80  78.791159  14.991159
4  Aberdeen City  S12000033  2015 to 2017  62.30  79.401786  17.101786
5  Aberdeen City  S12000033  2016 to 2018  62.45  78.592118  16.142118
6  Aberdeen City  S12000033  2017 to 2019  61.75  79.718521  17.968521
7  Aberdeen City  S12000033  2018 to 2020  59.60  78.982099  19.382099
8  Aberdeen City  S12000033  2019 to 2021  60.60  79.256410  18.656410
9  Aberdeen City  S12000033  2020 to 2022  59.30  79.122932  19.822932


In [3]:
# Create a pivot table showing HLE Gap over time for each area
df_trends = df_persons.pivot_table(
    index=['Area name', 'Area code'],
    columns='Period',
    values='HLE_Gap'
).reset_index()

print("✓ Pivoted data created")
print(f"✓ Rows: {len(df_trends)} (one per local authority)")
print(f"✓ Columns: {len(df_trends.columns)} (Area info + time periods)")

print("\nFirst few areas:")
print(df_trends.head())

✓ Pivoted data created
✓ Rows: 308 (one per local authority)
✓ Columns: 14 (Area info + time periods)

First few areas:
Period                              Area name  Area code  2011 to 2013  \
0                               Aberdeen City  S12000033     15.064634   
1                               Aberdeenshire  S12000034     14.040273   
2       Aneurin Bevan University Health Board  W11000028     20.030617   
3                                       Angus  S12000041     15.547531   
4                     Antrim and Newtownabbey  N09000001           NaN   

Period  2012 to 2014  2013 to 2015  2014 to 2016  2015 to 2017  2016 to 2018  \
0          14.196839     14.239290     14.991159     17.101786     16.142118   
1          13.752846     14.583363     12.915167     12.913749     13.048618   
2          19.642120     20.089829     20.908904     21.311111     20.811111   
3          14.398855     14.496973     14.385639     16.433238     17.301461   
4                NaN           NaN 

In [4]:
# Count how many time periods each area has data for
df_trends['periods_with_data'] = df_trends.iloc[:, 2:].notna().sum(axis=1)

print("Distribution of data completeness:")
print(df_trends['periods_with_data'].value_counts().sort_index(ascending=False))

print("\n" + "="*50)

# How many areas have ALL 12 periods?
complete_areas = df_trends[df_trends['periods_with_data'] == 12]
print(f"✓ Areas with complete data (all 12 periods): {len(complete_areas)}")

# Show sample of incomplete areas
incomplete = df_trends[df_trends['periods_with_data'] < 12]
print(f"\n✗ Areas with missing data: {len(incomplete)}")
print("\nSample of incomplete areas:")
print(incomplete[['Area name', 'periods_with_data']].head(10))

Distribution of data completeness:
periods_with_data
12    297
9      11
Name: count, dtype: int64

✓ Areas with complete data (all 12 periods): 297

✗ Areas with missing data: 11

Sample of incomplete areas:
Period                             Area name  periods_with_data
4                    Antrim and Newtownabbey                  9
5                        Ards and North Down                  9
7       Armagh City, Banbridge and Craigavon                  9
13                                   Belfast                  9
39                  Causeway Coast and Glens                  9
57                   Derry City and Strabane                  9
79                       Fermanagh and Omagh                  9
122                  Lisburn and Castlereagh                  9
131                               Mid Ulster                  9
132                      Mid and East Antrim                  9


In [5]:
# Filter to England only
df_england = df_complete[df_complete['Area code'].str.startswith('E')].copy()

print(f"✓ Filtered to England only: {len(df_england)} local authorities")
print("\nSample of English local authorities:")
print(df_england[['Area name', 'Area code']].head(10))

print("\n" + "="*50)
print("Geographic distribution check:")
print(df_england['Area name'].head(20))

NameError: name 'df_complete' is not defined

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Load the cleaned data
df = pd.read_csv('../outputs/hle_at_birth_cleaned.csv')

# Create "Persons" data (average of Male and Female)
df_persons = df.groupby(['Area name', 'Area code', 'Period']).agg({
    'HLE': 'mean',
    'LE': 'mean',
    'HLE_Gap': 'mean'
}).reset_index()

# Pivot to see trends over time
df_trends = df_persons.pivot_table(
    index=['Area name', 'Area code'],
    columns='Period',
    values='HLE_Gap'
).reset_index()

# Count complete data
df_trends['periods_with_data'] = df_trends.iloc[:, 2:].notna().sum(axis=1)

# Keep only complete areas
df_complete = df_trends[df_trends['periods_with_data'] == 12].copy()
df_complete = df_complete.drop(columns=['periods_with_data'])

# Filter to England only
df_england = df_complete[df_complete['Area code'].str.startswith('E')].copy()

print(f"✓ Loaded and processed data")
print(f"✓ England local authorities: {len(df_england)}")
print("\nFirst 10 English areas:")
print(df_england[['Area name', 'Area code']].head(10))

✓ Loaded and processed data
✓ England local authorities: 218

First 10 English areas:
Period                     Area name  Area code
8               Barking and Dagenham  E09000002
9                             Barnet  E09000003
10                          Barnsley  E08000038
11      Bath and North East Somerset  E06000022
12                           Bedford  E06000055
15                            Bexley  E09000004
16                        Birmingham  E08000025
17             Blackburn with Darwen  E06000008
18                         Blackpool  E06000009
20                            Bolton  E08000001


In [7]:
# Get the earliest and latest periods
earliest = '2011 to 2013'
latest = '2022 to 2024'

# Calculate the change in HLE Gap
df_england['gap_2011_2013'] = df_england[earliest]
df_england['gap_2022_2024'] = df_england[latest]
df_england['total_change'] = df_england['gap_2022_2024'] - df_england['gap_2011_2013']

# Calculate annual rate of change (11 years from 2012 to 2023)
years_elapsed = 11
df_england['annual_change'] = df_england['total_change'] / years_elapsed

print("✓ Calculated change over time")
print("\nTop 10 IMPROVING areas (gap getting smaller = better):")
improving = df_england.nsmallest(10, 'total_change')[['Area name', 'gap_2011_2013', 'gap_2022_2024', 'total_change']]
print(improving)

print("\n" + "="*50)

print("\nTop 10 WORSENING areas (gap getting bigger = worse):")
worsening = df_england.nlargest(10, 'total_change')[['Area name', 'gap_2011_2013', 'gap_2022_2024', 'total_change']]
print(worsening)

print("\n" + "="*50)
print(f"\nNational average change: {df_england['total_change'].mean():.2f} years")

✓ Calculated change over time

Top 10 IMPROVING areas (gap getting smaller = better):
Period               Area name  gap_2011_2013  gap_2022_2024  total_change
265                     Sutton      18.139474      14.562169     -3.577304
50                     Croydon      18.349867      15.268223     -3.081644
282             Waltham Forest      19.989119      17.186203     -2.802916
91      Hammersmith and Fulham      18.320833      15.697873     -2.622961
197                     Newham      21.015205      18.863502     -2.151703
275              Tower Hamlets      23.263530      21.484878     -1.778652
196        Newcastle upon Tyne      20.860731      19.167360     -1.693371
255                  Southwark      20.279550      18.770253     -1.509297
243                     Slough      20.567304      19.275974     -1.291330
24                       Brent      18.833544      17.859552     -0.973993


Top 10 WORSENING areas (gap getting bigger = worse):
Period                 Area name  

In [8]:
# Save the trend analysis
df_england.to_csv('../outputs/england_hle_trends.csv', index=False)

print("✓ Saved trend analysis to outputs/england_hle_trends.csv")
print(f"\n📊 Summary Statistics:")
print(f"   Areas improving (gap shrinking): {(df_england['total_change'] < 0).sum()}")
print(f"   Areas worsening (gap widening): {(df_england['total_change'] > 0).sum()}")
print(f"   National average change: {df_england['total_change'].mean():.2f} years")

✓ Saved trend analysis to outputs/england_hle_trends.csv

📊 Summary Statistics:
   Areas improving (gap shrinking): 20
   Areas worsening (gap widening): 198
   National average change: 2.73 years
